# TradeLab Signals and Indicators Plotting Demo

This notebook shows how to:
1. Download market data and plot candlesticks.
2. Compute and plot the `HeikinAshi` signal.
3. Compute and plot the `RSI` indicator.
4. Compute and plot the `EMA` indicator from the `HeikinAshi` signal.

In [7]:
from pathlib import Path
import sys

import pandas as pd
import yfinance as yf
import plotly.graph_objects as go

# Allow running from the repository without installing the package.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from trade_lab.signals import HeikinAshi
from trade_lab.indicators import RSI
from trade_lab.indicators import EMA

In [8]:
# Config
ticker = "SPY"
start = "2025-01-01"
end = "2026-01-01"

df = yf.download(ticker, start=start, end=end)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel("Ticker")

df = df.dropna().copy()
df.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2025-12-24,690.380005,690.830017,687.799988,687.950012,39445600
2025-12-26,690.309998,691.659973,689.270020,690.640015,41613300
2025-12-29,687.849976,689.200012,686.070007,687.539978,62559500
2025-12-30,687.010010,688.559998,686.580017,687.450012,47160700
2025-12-31,681.919983,687.359985,681.710022,687.140015,74144800


## 1) Price Candlesticks

In [9]:
price_fig = go.Figure(
    data=[
        go.Candlestick(
            x=df.index,
            open=df["Open"],
            high=df["High"],
            low=df["Low"],
            close=df["Close"],
            name=ticker,
        )
    ]
)
price_fig.update_layout(
    title=f"{ticker} Candlestick ({start} to {end})",
    xaxis_title="Date",
    yaxis_title="Price",
)
price_fig.show()

## 2) Heikin-Ashi Signal

In [10]:
ha_signal = HeikinAshi()
df_ha = ha_signal.compute(df.copy())
ha_signal.plot(ha_signal.raw_df)

df_ha[ha_signal.output_columns].tail()

Price,signal__ha_open,signal__ha_high,signal__ha_low,signal__ha_close
Date,,,,
2025-12-24,-0.004007,0.007034,-0.004007,0.004730
2025-12-26,-0.004359,0.003505,-0.004359,0.001783
2025-12-29,-0.003066,-0.001841,-0.006393,-0.004071
2025-12-30,0.000502,0.001301,-0.001579,-0.000385
2025-12-31,0.000444,0.000444,-0.008312,-0.004180


## 3) RSI Indicator

In [11]:
rsi = RSI(period=14)
df_rsi = rsi.compute(df.copy())
rsi.plot(df_rsi)

df_rsi[rsi.output_columns].tail()

Price,indicator__rsi_14
Date,
2025-12-24,61.844894
2025-12-26,61.758819
2025-12-29,58.668828
2025-12-30,57.608950
2025-12-31,51.533469


## 4) Exponential Moving Average from Heikin-Ashi signal.

In [12]:
ema = EMA(ha_signal, period=30)
df_ema = ema.compute(df.copy())
ema.plot(df_ema)